In [1]:
# imports
import pandas as pd
import numpy as np
import xgboost as xgb
from xgboost import XGBClassifier
import sklearn

In [ ]:
### Data

#read csv
training = pd.read_csv("train_data_m.csv")

#features
features = ['seed_dif', 'massey_rank_dif', 'avg_margin_dif',
       'avg_eff_A', 'avg_opp_eff_A', 'avg_eff_B', 'avg_opp_eff_B',
       'avg_thr_per_A', 'ft_per_A', 'avg_fg_per_A', 'avg_fg_a_per_A',
       'avg_thr_a_per_A', 'avg_to_per_A', 'avg_blk_per_A',
       'avg_opp_fg_a_per_A', 'avg_opp_fg_per_A', 'avg_opp_to_per_A',
       'avg_thr_per_B', 'ft_per_B', 'avg_fg_per_B', 'avg_fg_a_per_B',
       'avg_thr_a_per_B', 'avg_to_per_B', 'avg_blk_per_B',
       'avg_opp_fg_a_per_B', 'avg_opp_fg_per_B', 'avg_opp_to_per_B',
       'thr_a_per_dif', 'tempo_pred', 'pred_fg_per_dif', 'pred_or_per_dif']

training_dfs = []
val_dfs = []
val_years = []

years = [2015,2016,2017,2018,2019,2021,2022,2023,2024]

for i in years:
    if i == 2019:
        val_year = 2021
    else:
        val_year = i+1

    temp_training = training.query("Season <= @i")
    temp_val = training.query("Season == @val_year")

    training_dfs.append(temp_training)
    val_dfs.append(temp_val)
    val_years.append(val_year)

### Model

val_error = []

#model
mod = XGBClassifier(n_estimators=600, max_depth=4, learning_rate=0.01, subsample = 0.6, colsample_bynode = 0.8, objective='binary:logistic', seed = 323)

for i in range(len(years)):
    X_train = training_dfs[i][features]
    y_train = training_dfs[i]['result']

    val = val_dfs[i].copy()

    X_val = val[features]
    y_val = val['result']

    #fit model
    mod.fit(X_train, y_train)
    
    #make predictions
    preds = mod.predict_proba(X_val)

    val['pred'] = preds[:,1]

    val['loss'] = (val['pred'] - val['result'])**2

    print(val_years[i], "Loss:", np.mean(val['loss']))
    val_error.append(np.mean(val['loss']))


#print(val_base.drop('loss', axis = 1).sort_values('pred', ascending = False).head(5))
#val_base.drop('loss', axis = 1).sort_values('pred', ascending = True).head(5)

2016 Loss: 0.2033597123621719
2017 Loss: 0.17630170631471545
2018 Loss: 0.1969060477284492
2019 Loss: 0.16695335518538015
2021 Loss: 0.22735868010250185
2022 Loss: 0.2412032883546274
2023 Loss: 0.1945756299410693
2024 Loss: 0.20859717677368664
2025 Loss: 0.1658786439119256


In [ ]:
### Combined